# Exploring the Dataset: Audit Log (internal_share)

**Goal:** Explore `internal_share/logs/audit/audit.log` (Linux auditd) and define a parsed staging model (`stg_audit_line_raw`) for later schema finalization.

This notebook walks through:
1. Loading `internal_share/logs/audit/audit.log` (732 lines, Linux auditd format)
2. Parsing the two-level structure (outer key-value pairs + nested `msg='...'` blob)
3. Exploring all fields at every nesting level
4. Building a parsed staging DataFrame and a separate analysis DataFrame
5. Integrating ground truth labels (2 labeled lines - exfiltration service)
6. Mapping to SQL schema with PostgreSQL and MySQL types
7. Checking normalization violations (1NF, 2NF, 3NF) per the normalization rules checklist

---

**Dataset:** AIT Log Data Set V2.0 - russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064  
**Host:** internal_share (file server, internal network zone)  
**Related:** This log uses the same auditd format as `intranet_server/audit/audit.log` (DAT-42). This file shares the auditd source format with the intranet audit log and will be reconciled in the final audit schema design.

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
├── data-201-security-log-analysis/   <-- this repo
│   └── notebooks/                    <-- this notebook is here
└── russellmitchell/                  <-- dataset is here
```

If your dataset is somewhere else, just change `DATASET_ROOT` below.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

AUDIT_LOG = DATASET_ROOT / "gather" / "internal_share" / "logs" / "audit" / "audit.log"
LABEL_FILE = DATASET_ROOT / "labels" / "internal_share" / "logs" / "audit" / "audit.log"

# Verify paths
for label, path in [
    ("Dataset root", DATASET_ROOT),
    ("Audit log", AUDIT_LOG),
    ("Label file", LABEL_FILE),
]:
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"{label}: {path.resolve()} [{status}]")

Dataset root: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell [FOUND]
Audit log: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell/gather/internal_share/logs/audit/audit.log [FOUND]
Label file: /Users/namansudan/Desktop/sjsu/classes/data-201/russellmitchell/labels/internal_share/logs/audit/audit.log [FOUND]


## 1. Load Raw Data

The file `internal_share/logs/audit/audit.log` contains Linux kernel audit events from the file server.

### Auditd Format

Each line follows the format:
```
type=EVENT_TYPE msg=audit(EPOCH.MS:SERIAL): key=value ... [msg='key=value...']
```

Two nesting levels:
- **Outer:** `type`, `pid`, `uid`, `auid`, `ses`, and type-specific fields (like `old-auid`, `tty`, `arch`, `syscall`, etc.)
- **Inner (nested `msg='...'`):** PAM/service fields packed into a single quoted string (op, acct, exe, hostname, addr, terminal, res, unit, comm)

In [2]:
with open(AUDIT_LOG) as f:
    raw_lines = f.readlines()

print(f"Loaded {len(raw_lines)} lines from: {AUDIT_LOG.name}")
print("\nFirst 5 lines:")
for line in raw_lines[:5]:
    print(f"  {line.rstrip()}")
print("\nLast 3 lines:")
for line in raw_lines[-3:]:
    print(f"  {line.rstrip()}")

Loaded 732 lines from: audit.log

First 5 lines:
  type=USER_ACCT msg=audit(1642724221.475:149): pid=1716 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'
  type=CRED_ACQ msg=audit(1642724221.479:150): pid=1716 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:setcred acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'
  type=LOGIN msg=audit(1642724221.479:151): pid=1716 uid=0 old-auid=4294967295 auid=0 tty=(none) old-ses=4294967295 ses=36 res=1
  type=USER_START msg=audit(1642724221.483:152): pid=1716 uid=0 auid=0 ses=36 msg='op=PAM:session_open acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'
  type=CRED_DISP msg=audit(1642724221.491:153): pid=1716 uid=0 auid=0 ses=36 msg='op=PAM:setcred acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

Last 3 lines:
  type=USER_START msg=audit(1643066221.963:870): pid=15520 uid=0 

## 2. Parse Audit Log Format

### 2.1 Event Type Inventory

First, let's count every distinct event type before parsing.

In [3]:
import re
from collections import Counter

type_counts = Counter()
for line in raw_lines:
    m = re.match(r"type=(\S+)", line)
    if m:
        type_counts[m.group(1)] += 1

print(f"Distinct event types: {len(type_counts)}\n")
print(f"{'Event Type':<20} {'Count':>6} {'%':>7}")
print("-" * 35)
for event_type, count in type_counts.most_common():
    print(f"{event_type:<20} {count:>6} {count / len(raw_lines) * 100:>6.1f}%")

Distinct event types: 11

Event Type            Count       %
-----------------------------------
USER_ACCT               106   14.5%
CRED_ACQ                106   14.5%
LOGIN                   106   14.5%
USER_START              106   14.5%
CRED_DISP               106   14.5%
USER_END                106   14.5%
SERVICE_START            47    6.4%
SERVICE_STOP             37    5.1%
AVC                       4    0.5%
SYSCALL                   4    0.5%
PROCTITLE                 4    0.5%


### 2.2 Format Categories with Examples

Show one example line for each event type to understand the field structure.

In [4]:
# Show one example line per event type
seen_types = set()
for i, line in enumerate(raw_lines, 1):
    m = re.match(r"type=(\S+)", line)
    if m and m.group(1) not in seen_types:
        seen_types.add(m.group(1))
        print(f"[line {i}] {line.rstrip()}")
        print()
print(f"Showed examples for all {len(seen_types)} event types.")

[line 1] type=USER_ACCT msg=audit(1642724221.475:149): pid=1716 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

[line 2] type=CRED_ACQ msg=audit(1642724221.479:150): pid=1716 uid=0 auid=4294967295 ses=4294967295 msg='op=PAM:setcred acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

[line 3] type=LOGIN msg=audit(1642724221.479:151): pid=1716 uid=0 old-auid=4294967295 auid=0 tty=(none) old-ses=4294967295 ses=36 res=1

[line 4] type=USER_START msg=audit(1642724221.483:152): pid=1716 uid=0 auid=0 ses=36 msg='op=PAM:session_open acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

[line 5] type=CRED_DISP msg=audit(1642724221.491:153): pid=1716 uid=0 auid=0 ses=36 msg='op=PAM:setcred acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success'

[line 6] type=USER_END msg=audit(1642724221.491:154): pid=1716 uid=0 auid=0 ses=36 msg='o

### 2.3 Format Categories

Based on the examples above, the 11 event types fall into 3 format categories:

| Category | Event Types | Count | Key Fields |
|----------|------------|------:|------------|
| **PAM events** (with `msg='...'`) | USER_ACCT, CRED_ACQ, USER_START, CRED_DISP, USER_END | 530 | pid, uid, auid, ses + nested msg (op, acct, exe, hostname, addr, terminal, res) |
| **LOGIN** (no nested msg) | LOGIN | 106 | pid, uid, old-auid, auid, tty, old-ses, ses, res |
| **SERVICE** (with `msg='...'`) | SERVICE_START, SERVICE_STOP | 84 | pid, uid, auid, ses + nested msg (unit, comm, exe, hostname, addr, terminal, res) |
| **Kernel** (no nested msg) | AVC, SYSCALL, PROCTITLE | 12 | Highly heterogeneous - each type has unique fields |

### 2.4 Parsing Strategy

Same two-pass approach as the intranet_server audit log (DAT-42):
1. Extract `type`, `epoch`, `serial`, `timestamp` from the audit header
2. If `msg='...'` present: store raw blob, parse nested key-value pairs as `msg_*` columns
3. Parse outer key-value pairs for all remaining fields

In [5]:
from datetime import UTC, datetime

import pandas as pd


def parse_audit_line(line_num, line):
    """Parse a single audit log line into a flat dictionary.

    Extracts:
    - Common fields: line_number, type, epoch timestamp, serial
    - Outer key-value pairs (pid, uid, auid, ses, etc.)
    - The raw msg='...' string as-is (for the raw table)
    - Nested msg='...' key-value pairs as msg_* columns (for analysis)
    """
    record = {"line_number": line_num}
    line = line.rstrip()
    record["raw_line"] = line

    # Extract type
    m_type = re.match(r"type=(\S+)", line)
    if not m_type:
        return record
    record["type"] = m_type.group(1)

    # Extract audit timestamp and serial: msg=audit(EPOCH.MS:SERIAL)
    m_ts = re.search(r"msg=audit\((\d+\.\d+):(\d+)\)", line)
    if m_ts:
        epoch = float(m_ts.group(1))
        record["epoch"] = epoch
        record["serial"] = int(m_ts.group(2))
        record["timestamp"] = datetime.fromtimestamp(epoch, tz=UTC)

    # Extract nested msg='...' content first (so we can remove it before parsing outer fields)
    m_nested = re.search(r"msg='([^']+)'", line)
    if m_nested:
        nested_msg = m_nested.group(1)
        # Store the raw msg string for the raw table
        record["msg"] = nested_msg
        # Parse nested key=value pairs for analysis columns
        for m in re.finditer(r'(\w+)="?([^"\'\'\s]+)"?', nested_msg):
            key = f"msg_{m.group(1)}"
            record[key] = m.group(2)

    # Parse outer key=value pairs (everything after the audit(...): prefix, excluding nested msg)
    outer = re.sub(r"^type=\S+\s+msg=audit\([^)]+\):\s*", "", line)
    if m_nested:
        outer = outer.replace(f"msg='{m_nested.group(1)}'", "")

    for m in re.finditer(r'([\w-]+)=("[^"]*"|\([^)]*\)|\S+)', outer):
        key = m.group(1).replace("-", "_")
        val = m.group(2).strip('"')
        if key not in record:
            record[key] = val

    return record


# Parse all lines
parsed = [parse_audit_line(i, line) for i, line in enumerate(raw_lines, 1)]
print(f"Parsed {len(parsed)} records")
print(f"Sample (line 1): {parsed[0]}")

Parsed 732 records
Sample (line 1): {'line_number': 1, 'raw_line': 'type=USER_ACCT msg=audit(1642724221.475:149): pid=1716 uid=0 auid=4294967295 ses=4294967295 msg=\'op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success\'', 'type': 'USER_ACCT', 'epoch': 1642724221.475, 'serial': 149, 'timestamp': datetime.datetime(2022, 1, 21, 0, 17, 1, 475000, tzinfo=datetime.timezone.utc), 'msg': 'op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success', 'msg_op': 'PAM:accounting', 'msg_acct': 'root', 'msg_exe': '/usr/sbin/cron', 'msg_hostname': '?', 'msg_addr': '?', 'msg_terminal': 'cron', 'msg_res': 'success', 'pid': '1716', 'uid': '0', 'auid': '4294967295', 'ses': '4294967295'}


In [6]:
# Build the analysis DataFrame with all columns (including msg_* parsed sub-fields)
df = pd.DataFrame(parsed)

print(f"DataFrame shape: {df.shape}")
print(f"\nAll columns ({len(df.columns)}):\n")
print(f"{'Column':<25} {'Non-null':>10} {'Null':>8} {'Unique':>8} {'Dtype':>15}")
print("-" * 70)
for col in df.columns:
    non_null = df[col].notna().sum()
    null = df[col].isna().sum()
    unique = df[col].nunique()
    dtype = str(df[col].dtype)
    print(f"{col:<25} {non_null:>10} {null:>8} {unique:>8} {dtype:>15}")

DataFrame shape: (732, 49)

All columns (49):

Column                      Non-null     Null   Unique           Dtype
----------------------------------------------------------------------
line_number                      732        0      732           int64
raw_line                         732        0      732          object
type                             732        0       11          object
epoch                            732        0      360         float64
serial                           732        0      724           int64
timestamp                        732        0      360 datetime64[ns, UTC]
msg                              614      118       20          object
msg_op                           530      202        4          object
msg_acct                         530      202        1          object
msg_exe                          614      118        2          object
msg_hostname                     614      118        1          object
msg_addr                  

## 3. Field-by-Field Exploration

### 3.1 Event Type Distribution

In [7]:
# Categorize event types
pam_types = {"USER_ACCT", "CRED_ACQ", "USER_START", "CRED_DISP", "USER_END"}
login_types = {"LOGIN"}
service_types = {"SERVICE_START", "SERVICE_STOP"}
kernel_types = {"AVC", "SYSCALL", "PROCTITLE"}

type_dist = df["type"].value_counts()

print("=== Event Type Distribution ===\n")
for category, types in [
    ("PAM", pam_types),
    ("Login", login_types),
    ("Service", service_types),
    ("Kernel", kernel_types),
]:
    cat_total = sum(type_dist.get(t, 0) for t in types)
    print(f"{category}: {cat_total} rows ({cat_total / len(df) * 100:.1f}%)")
    for t in sorted(types):
        count = type_dist.get(t, 0)
        print(f"  {t:<20} {count:>5} ({count / len(df) * 100:.1f}%)")
    print()

=== Event Type Distribution ===

PAM: 530 rows (72.4%)
  CRED_ACQ               106 (14.5%)
  CRED_DISP              106 (14.5%)
  USER_ACCT              106 (14.5%)
  USER_END               106 (14.5%)
  USER_START             106 (14.5%)

Login: 106 rows (14.5%)
  LOGIN                  106 (14.5%)

Service: 84 rows (11.5%)
  SERVICE_START           47 (6.4%)
  SERVICE_STOP            37 (5.1%)

Kernel: 12 rows (1.6%)
  AVC                      4 (0.5%)
  PROCTITLE                4 (0.5%)
  SYSCALL                  4 (0.5%)



### 3.2 Timestamp and Serial

In [8]:
print("=== Timestamp Range ===")
print(f"  Earliest: {df['timestamp'].min()}")
print(f"  Latest:   {df['timestamp'].max()}")
print(f"  Span:     {df['timestamp'].max() - df['timestamp'].min()}")

print("\n=== Serial ===")
serial_unique = df["serial"].nunique()
print(f"  Range: {df['serial'].min()} - {df['serial'].max()}")
print(f"  Unique: {serial_unique} (out of {len(df)} lines)")
print(f"  Duplicates: {len(df) - serial_unique} lines share a serial with another line")

# Show duplicate serials (multi-line events)
serial_counts = df["serial"].value_counts()
multi_line = serial_counts[serial_counts > 1]
if len(multi_line) > 0:
    print(f"\n  Multi-line serials ({len(multi_line)}):")
    for serial, count in multi_line.items():
        types_in_serial = df[df["serial"] == serial]["type"].tolist()
        print(f"    serial={serial}: {count} lines -> {types_in_serial}")
else:
    print("\n  No multi-line events (all serials unique).")

=== Timestamp Range ===
  Earliest: 2022-01-21 00:17:01.475000+00:00
  Latest:   2022-01-24 23:17:01.967000+00:00
  Span:     3 days 23:00:00.492000

=== Serial ===
  Range: 149 - 872
  Unique: 724 (out of 732 lines)
  Duplicates: 8 lines share a serial with another line

  Multi-line serials (4):
    serial=196: 3 lines -> ['AVC', 'SYSCALL', 'PROCTITLE']
    serial=195: 3 lines -> ['AVC', 'SYSCALL', 'PROCTITLE']
    serial=194: 3 lines -> ['AVC', 'SYSCALL', 'PROCTITLE']
    serial=193: 3 lines -> ['AVC', 'SYSCALL', 'PROCTITLE']


In [9]:
# Events per day
df["date"] = df["timestamp"].dt.date
print("=== Events per Day ===\n")
for date, count in df.groupby("date").size().items():
    print(f"  {date}: {count} events")

=== Events per Day ===

  2022-01-21: 210 events
  2022-01-22: 168 events
  2022-01-23: 182 events
  2022-01-24: 172 events


### 3.3 Process and User IDs: pid, uid, auid, ses

In [10]:
for col in ["pid", "uid", "auid", "ses"]:
    print(f"=== {col} ===")
    present = df[col].notna().sum()
    unique = df[col].nunique()
    print(f"  Present: {present}/{len(df)} ({present / len(df) * 100:.1f}%)")
    print(f"  Unique values: {unique}")
    if unique <= 10:
        # Show all values with counts
        vc = df[col].value_counts(dropna=False)
        for val, count in vc.items():
            print(f"    {val}: {count} ({count / len(df) * 100:.1f}%)")
    else:
        # Show top values and range
        numeric_vals = pd.to_numeric(df[col], errors="coerce")
        print(f"  Range: {numeric_vals.min()} - {numeric_vals.max()}")
        print("  Top 5 values:")
        for val, count in df[col].value_counts().head(5).items():
            print(f"    {val}: {count}")
    # Check for sentinel value 4294967295
    sentinel_count = (df[col] == "4294967295").sum() + (df[col] == 4294967295).sum()
    if sentinel_count > 0:
        print(
            f"  ** Sentinel 4294967295 (unset): {sentinel_count} rows ({sentinel_count / len(df) * 100:.1f}%) **"
        )
    print()

=== pid ===
  Present: 728/732 (99.5%)
  Unique values: 108
  Range: 1.0 - 32269.0
  Top 5 values:
    1: 84
    30310: 8
    3398: 6
    761: 6
    30391: 6

=== uid ===
  Present: 724/732 (98.9%)
  Unique values: 1
    0: 724 (98.9%)
    nan: 8 (1.1%)

=== auid ===
  Present: 724/732 (98.9%)
  Unique values: 2
    0: 424 (57.9%)
    4294967295: 300 (41.0%)
    nan: 8 (1.1%)
  ** Sentinel 4294967295 (unset): 300 rows (41.0%) **

=== ses ===
  Present: 724/732 (98.9%)
  Unique values: 107
  Range: 36.0 - 4294967295.0
  Top 5 values:
    4294967295: 300
    115: 4
    113: 4
    112: 4
    111: 4
  ** Sentinel 4294967295 (unset): 300 rows (41.0%) **



### 3.4 Nested msg Fields (1NF Violation)

The `msg` column packs multiple key-value pairs into a single TEXT blob. This is a 1NF violation. Let's explore the parsed sub-fields.

In [11]:
msg_cols = [c for c in df.columns if c.startswith("msg_")]
print(f"Nested msg sub-fields ({len(msg_cols)}): {msg_cols}\n")

for col in msg_cols:
    present = df[col].notna().sum()
    unique = df[col].nunique()
    print(f"=== {col} ===")
    print(f"  Present: {present}/{len(df)} ({present / len(df) * 100:.1f}%)")
    print(f"  Unique values: {unique}")
    if unique <= 20:
        for val, count in df[col].value_counts().items():
            print(f"    {val}: {count}")
    else:
        for val, count in df[col].value_counts().head(5).items():
            print(f"    {val}: {count}")
        print(f"    ... and {unique - 5} more")
    print()

Nested msg sub-fields (9): ['msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'msg_unit', 'msg_comm']

=== msg_op ===
  Present: 530/732 (72.4%)
  Unique values: 4
    PAM:setcred: 212
    PAM:accounting: 106
    PAM:session_open: 106
    PAM:session_close: 106

=== msg_acct ===
  Present: 530/732 (72.4%)
  Unique values: 1
    root: 530

=== msg_exe ===
  Present: 614/732 (83.9%)
  Unique values: 2
    /usr/sbin/cron: 530
    /lib/systemd/systemd: 84

=== msg_hostname ===
  Present: 614/732 (83.9%)
  Unique values: 1
    ?: 614

=== msg_addr ===
  Present: 614/732 (83.9%)
  Unique values: 1
    ?: 614

=== msg_terminal ===
  Present: 614/732 (83.9%)
  Unique values: 2
    cron: 530
    ?: 84

=== msg_res ===
  Present: 614/732 (83.9%)
  Unique values: 1
    success: 614

=== msg_unit ===
  Present: 84/732 (11.5%)
  Unique values: 16
    apt-daily: 18
    motd-news: 16
    apt-daily-upgrade: 8
    systemd-tmpfiles-clean: 8
    apport: 3
    systemd

### 3.5 Field Presence Per Event Type

Each event type uses a different subset of columns. Event type strongly correlates with which fields are populated, which indicates structural heterogeneity and may motivate subtype modeling in the final design.

In [12]:
# Show which columns are populated for each event type
skip_cols = {"line_number", "type", "epoch", "serial", "timestamp", "date"}
data_cols = [c for c in df.columns if c not in skip_cols]

print("=== Columns populated per event type ===\n")
for event_type in df["type"].value_counts().index:
    subset = df[df["type"] == event_type]
    populated = [c for c in data_cols if subset[c].notna().any()]
    print(f"{event_type} ({len(subset)} rows):")
    print(f"  {populated}")
    print()

=== Columns populated per event type ===

USER_ACCT (106 rows):
  ['raw_line', 'msg', 'msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'pid', 'uid', 'auid', 'ses']

CRED_ACQ (106 rows):
  ['raw_line', 'msg', 'msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'pid', 'uid', 'auid', 'ses']

LOGIN (106 rows):
  ['raw_line', 'pid', 'uid', 'auid', 'ses', 'old_auid', 'tty', 'old_ses', 'res']

USER_START (106 rows):
  ['raw_line', 'msg', 'msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'pid', 'uid', 'auid', 'ses']

CRED_DISP (106 rows):
  ['raw_line', 'msg', 'msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'pid', 'uid', 'auid', 'ses']

USER_END (106 rows):
  ['raw_line', 'msg', 'msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'pid', 'uid', 'auid', 'ses']

SERVICE_START (47 rows):
  ['raw_line', 'msg', 'm

### 3.6 LOGIN-Specific Fields

In [13]:
df_login = df[df["type"] == "LOGIN"]
print(f"LOGIN events: {len(df_login)} rows\n")

for col in ["old_auid", "old_ses", "tty", "res"]:
    if col in df_login.columns:
        unique_vals = df_login[col].dropna().unique()
        print(f"  {col}: {list(unique_vals)}")

print("\nSample LOGIN records:")
login_cols = [
    "line_number",
    "type",
    "timestamp",
    "pid",
    "uid",
    "old_auid",
    "auid",
    "tty",
    "old_ses",
    "ses",
    "res",
]
login_cols_present = [c for c in login_cols if c in df_login.columns]
print(df_login[login_cols_present].head(5).to_string(index=False))

LOGIN events: 106 rows

  old_auid: ['4294967295']
  old_ses: ['4294967295']
  tty: ['(none)']
  res: ['1']

Sample LOGIN records:
 line_number  type                        timestamp   pid uid   old_auid auid    tty    old_ses ses res
           3 LOGIN 2022-01-21 00:17:01.479000+00:00  1716   0 4294967295    0 (none) 4294967295  36   1
          11 LOGIN 2022-01-21 01:17:01.785000+00:00  4462   0 4294967295    0 (none) 4294967295  37   1
          17 LOGIN 2022-01-21 02:17:01.851000+00:00  7159   0 4294967295    0 (none) 4294967295  38   1
          23 LOGIN 2022-01-21 03:17:01.899000+00:00  9818   0 4294967295    0 (none) 4294967295  39   1
          29 LOGIN 2022-01-21 04:17:01.947000+00:00 12485   0 4294967295    0 (none) 4294967295  40   1


### 3.7 SERVICE-Specific Fields

In [14]:
df_service = df[df["type"].isin({"SERVICE_START", "SERVICE_STOP"})]
print(f"SERVICE events: {len(df_service)} rows\n")

# Show unit distribution (this is where the exfiltration service appears)
print("=== msg_unit distribution ===")
if "msg_unit" in df_service.columns:
    for val, count in df_service["msg_unit"].value_counts().items():
        print(f"  {val}: {count}")

print("\n=== msg_comm distribution ===")
if "msg_comm" in df_service.columns:
    for val, count in df_service["msg_comm"].value_counts().items():
        print(f"  {val}: {count}")

print("\n=== msg_exe distribution ===")
if "msg_exe" in df_service.columns:
    for val, count in df_service["msg_exe"].value_counts().items():
        print(f"  {val}: {count}")

print("\n=== msg_res distribution ===")
if "msg_res" in df_service.columns:
    for val, count in df_service["msg_res"].value_counts().items():
        print(f"  {val}: {count}")

print("\nSample SERVICE records:")
svc_cols = [
    "line_number",
    "type",
    "timestamp",
    "pid",
    "uid",
    "msg_unit",
    "msg_comm",
    "msg_exe",
    "msg_res",
]
svc_cols_present = [c for c in svc_cols if c in df_service.columns]
print(df_service[svc_cols_present].head(10).to_string(index=False))

SERVICE events: 84 rows

=== msg_unit distribution ===
  apt-daily: 18
  motd-news: 16
  apt-daily-upgrade: 8
  systemd-tmpfiles-clean: 8
  apport: 3
  systemd-udevd: 3
  ssh: 3
  systemd-networkd: 3
  systemd-resolved: 3
  systemd-hostnamed: 3
  systemd-timesyncd: 3
  systemd-journal-flush: 3
  systemd-journald: 3
  grub-common: 3
  fstrim: 2
  put: 2

=== msg_comm distribution ===
  systemd: 84

=== msg_exe distribution ===
  /lib/systemd/systemd: 84

=== msg_res distribution ===
  success: 84

Sample SERVICE records:
 line_number          type                        timestamp pid uid      msg_unit msg_comm              msg_exe msg_res
           7 SERVICE_START 2022-01-21 00:18:36.239000+00:00   1   0     apt-daily  systemd /lib/systemd/systemd success
           8  SERVICE_STOP 2022-01-21 00:18:36.239000+00:00   1   0     apt-daily  systemd /lib/systemd/systemd success
          57 SERVICE_START 2022-01-21 06:19:57.759000+00:00   1   0        apport  systemd /lib/systemd/systemd su

### 3.8 AVC, SYSCALL, PROCTITLE (Kernel Events)

In [15]:
# AVC events
df_avc = df[df["type"] == "AVC"]
print(f"AVC events: {len(df_avc)} rows")
if len(df_avc) > 0:
    for col in ["apparmor", "operation", "profile", "name", "comm"]:
        if col in df_avc.columns and df_avc[col].notna().any():
            print(f"  {col}: {list(df_avc[col].dropna().unique())}")

print()

# SYSCALL events
df_syscall = df[df["type"] == "SYSCALL"]
print(f"SYSCALL events: {len(df_syscall)} rows")
if len(df_syscall) > 0:
    for col in ["arch", "syscall", "success", "exit", "comm", "exe", "tty"]:
        if col in df_syscall.columns and df_syscall[col].notna().any():
            print(f"  {col}: {list(df_syscall[col].dropna().unique())}")

print()

# PROCTITLE events
df_proctitle = df[df["type"] == "PROCTITLE"]
print(f"PROCTITLE events: {len(df_proctitle)} rows")
if len(df_proctitle) > 0 and "proctitle" in df_proctitle.columns:
    for val in df_proctitle["proctitle"].dropna().unique():
        # Decode hex to ASCII
        try:
            decoded = bytes.fromhex(val).decode("ascii", errors="replace").replace("\x00", " ")
        except ValueError:
            decoded = "(decode failed)"
        print(f"  hex: {val[:60]}...")
        print(f"  decoded: {decoded}")

AVC events: 4 rows
  apparmor: ['STATUS']
  operation: ['profile_replace']
  profile: ['unconfined']
  name: ['/sbin/dhclient', '/usr/lib/NetworkManager/nm-dhcp-client.action', '/usr/lib/NetworkManager/nm-dhcp-helper', '/usr/lib/connman/scripts/dhclient-script']
  comm: ['apparmor_parser']

SYSCALL events: 4 rows
  arch: ['c000003e']
  syscall: ['1']
  success: ['yes']
  exit: ['44953', '24993', '25025', '23625']
  comm: ['apparmor_parser']
  exe: ['/sbin/apparmor_parser']
  tty: ['pts0']

PROCTITLE events: 4 rows
  hex: 61707061726D6F725F706172736572002D72002D54002D57002F6574632F...
  decoded: apparmor_parser -r -T -W /etc/apparmor.d/sbin.dhclient


### 3.9 Multi-Line Events (Serial Grouping)

In [16]:
serial_counts = df["serial"].value_counts()
multi_line = serial_counts[serial_counts > 1]

print(f"Total serials: {df['serial'].nunique()}")
print(f"Serials with 1 line: {(serial_counts == 1).sum()}")
print(f"Serials with >1 line: {len(multi_line)}")

if len(multi_line) > 0:
    print("\n=== Multi-line event groups ===")
    for serial in sorted(multi_line.index):
        group = df[df["serial"] == serial]
        types = group["type"].tolist()
        lines = group["line_number"].tolist()
        print(f"  serial={serial}: lines {lines} -> {types}")

Total serials: 724
Serials with 1 line: 720
Serials with >1 line: 4

=== Multi-line event groups ===
  serial=193: lines [45, 46, 47] -> ['AVC', 'SYSCALL', 'PROCTITLE']
  serial=194: lines [48, 49, 50] -> ['AVC', 'SYSCALL', 'PROCTITLE']
  serial=195: lines [51, 52, 53] -> ['AVC', 'SYSCALL', 'PROCTITLE']
  serial=196: lines [54, 55, 56] -> ['AVC', 'SYSCALL', 'PROCTITLE']


### 3.10 Session Analysis

In [17]:
# Convert ses to numeric for analysis
ses_numeric = pd.to_numeric(df["ses"], errors="coerce")
unset_mask = ses_numeric == 4294967295

print("=== Session (ses) Field ===")
print(f"  Total rows with ses: {ses_numeric.notna().sum()}")
print(f"  Unique sessions: {ses_numeric.nunique()}")
print(f"  Unset (4294967295): {unset_mask.sum()} rows ({unset_mask.sum() / len(df) * 100:.1f}%)")

# Named sessions (excluding unset)
named_sessions = ses_numeric[~unset_mask & ses_numeric.notna()]
if len(named_sessions) > 0:
    print(f"  Named sessions: {named_sessions.nunique()} unique, {len(named_sessions)} rows")
    print(f"  Session range: {int(named_sessions.min())} - {int(named_sessions.max())}")
    # Events per session
    session_sizes = named_sessions.value_counts()
    print(
        f"  Events per session: min={session_sizes.min()}, max={session_sizes.max()}, median={session_sizes.median():.0f}"
    )

=== Session (ses) Field ===
  Total rows with ses: 724
  Unique sessions: 107
  Unset (4294967295): 300 rows (41.0%)
  Named sessions: 106 unique, 424 rows
  Session range: 36 - 141
  Events per session: min=4, max=4, median=4


### 3.11 Account (acct) Analysis

In [18]:
if "msg_acct" in df.columns:
    print("=== msg_acct (Account) ===")
    for val, count in df["msg_acct"].value_counts(dropna=False).items():
        label = val if val is not None and pd.notna(val) else "(null - no nested msg)"
        print(f"  {label}: {count} ({count / len(df) * 100:.1f}%)")
else:
    print("No msg_acct column found.")

=== msg_acct (Account) ===
  root: 530 (72.4%)
  (null - no nested msg): 202 (27.6%)


### 3.12 Executable (exe) Analysis

In [19]:
# msg_exe (from nested msg) and exe (from outer, e.g., SYSCALL)
for col in ["msg_exe", "exe"]:
    if col in df.columns and df[col].notna().any():
        print(f"=== {col} ===")
        for val, count in df[col].value_counts().items():
            print(f"  {val}: {count}")
        print()

=== msg_exe ===
  /usr/sbin/cron: 530
  /lib/systemd/systemd: 84

=== exe ===
  /sbin/apparmor_parser: 4



### 3.13 Result (res) Fields

In [20]:
# Two result fields: msg_res (from nested msg) and res (from outer, LOGIN)
print("=== msg_res (nested msg, PAM/SERVICE) ===")
if "msg_res" in df.columns:
    print(f"  Values: {dict(df['msg_res'].value_counts(dropna=False))}")

print("\n=== res (outer, LOGIN) ===")
if "res" in df.columns:
    print(f"  Values: {dict(df['res'].value_counts(dropna=False))}")

=== msg_res (nested msg, PAM/SERVICE) ===
  Values: {'success': np.int64(614), nan: np.int64(118)}

=== res (outer, LOGIN) ===
  Values: {nan: np.int64(626), '1': np.int64(106)}


### 3.14 Hostname and Address Fields

In [21]:
for col in ["msg_hostname", "msg_addr"]:
    if col in df.columns and df[col].notna().any():
        print(f"=== {col} ===")
        for val, count in df[col].value_counts().items():
            print(f"  {val}: {count}")
        print()

# Check if any real IP addresses appear (not just '?')
real_addrs = df[
    df.get("msg_hostname", pd.Series(dtype="object")).notna()
    & (df.get("msg_hostname", pd.Series(dtype="object")) != "?")
]
if len(real_addrs) > 0:
    print(f"Rows with real network addresses: {len(real_addrs)}")
else:
    print("No real network addresses found (all hostname=?, addr=?).")
    print(
        "This is expected: internal_share is a file server with only local cron/systemd activity."
    )
    print("No SSH logins or network-based authentication events appear in this log.")

=== msg_hostname ===
  ?: 614

=== msg_addr ===
  ?: 614

No real network addresses found (all hostname=?, addr=?).
This is expected: internal_share is a file server with only local cron/systemd activity.
No SSH logins or network-based authentication events appear in this log.


## 4. Parsed Staging DataFrame

Build the definitive parsed staging DataFrame that maps 1:1 to the SQL table. The `msg_*` analysis columns are dropped - the raw table stores `msg` as a single TEXT blob (1NF violation to be resolved during normalization).

In [22]:
# Build the raw 1:1 DataFrame for the raw table.
# The raw table stores msg as a single TEXT blob (not parsed into msg_* columns).
# The msg_* columns stay in df for analysis but are excluded from df_raw.
msg_star_cols = [c for c in df.columns if c.startswith("msg_")]
extra_cols = ["date"]  # derived column not in raw data
df_raw = df.drop(columns=msg_star_cols + extra_cols, errors="ignore")
df_raw.insert(0, "source_host", "internal_share")
df_raw.insert(1, "source_log", "audit.log")

print(f"Raw DataFrame shape: {df_raw.shape}")
print(f"Analysis DataFrame shape: {df.shape}")
print(f"Dropped msg_* columns: {msg_star_cols}")
print(f"Raw columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
print()
print("Data types:")
print(df_raw.dtypes.to_string())
print()
print("Null counts:")
nulls = df_raw.isnull().sum()
for col, n in nulls.items():
    if n > 0:
        print(f"  {col}: {n} nulls ({n / len(df_raw) * 100:.1f}%)")

Raw DataFrame shape: (732, 42)
Analysis DataFrame shape: (732, 50)
Dropped msg_* columns: ['msg_op', 'msg_acct', 'msg_exe', 'msg_hostname', 'msg_addr', 'msg_terminal', 'msg_res', 'msg_unit', 'msg_comm']
Raw columns (42): ['source_host', 'source_log', 'line_number', 'raw_line', 'type', 'epoch', 'serial', 'timestamp', 'msg', 'pid', 'uid', 'auid', 'ses', 'old_auid', 'tty', 'old_ses', 'res', 'apparmor', 'operation', 'profile', 'name', 'comm', 'arch', 'syscall', 'success', 'exit', 'a0', 'a1', 'a2', 'a3', 'items', 'ppid', 'gid', 'euid', 'suid', 'fsuid', 'egid', 'sgid', 'fsgid', 'exe', 'key', 'proctitle']

Data types:
source_host                 object
source_log                  object
line_number                  int64
raw_line                    object
type                        object
epoch                      float64
serial                       int64
timestamp      datetime64[ns, UTC]
msg                         object
pid                         object
uid                         obj

In [23]:
# Display first and last rows of the raw DataFrame
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
print("First 5 rows:")
display(df_raw.head())
print("\nLast 5 rows:")
display(df_raw.tail())

First 5 rows:


,source_host,source_log,line_number,raw_line,type,epoch,serial,timestamp,msg,pid,uid,auid,ses,old_auid,tty,old_ses,res,apparmor,operation,profile,name,comm,arch,syscall,success,exit,a0,a1,a2,a3,items,ppid,gid,euid,suid,fsuid,egid,sgid,fsgid,exe,key,proctitle
0,internal_share,audit.log,1,type=USER_ACCT msg=audit(1642724221.475:149): pid=1716 u...,USER_ACCT,1.642724e+09,149,2022-01-21 00:17:01.475000+00:00,"op=PAM:accounting acct=""root"" exe=""/usr/sbin/cron"" hostn...",1716,0,4294967295,4294967295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,internal_share,audit.log,2,type=CRED_ACQ msg=audit(1642724221.479:150): pid=1716 ui...,CRED_ACQ,1.642724e+09,150,2022-01-21 00:17:01.479000+00:00,"op=PAM:setcred acct=""root"" exe=""/usr/sbin/cron"" hostname...",1716,0,4294967295,4294967295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,internal_share,audit.log,3,type=LOGIN msg=audit(1642724221.479:151): pid=1716 uid=0...,LOGIN,1.642724e+09,151,2022-01-21 00:17:01.479000+00:00,NaN,1716,0,0,36,4294967295,(none),4294967295,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,internal_share,audit.log,4,type=USER_START msg=audit(1642724221.483:152): pid=1716 ...,USER_START,1.642724e+09,152,2022-01-21 00:17:01.483000+00:00,"op=PAM:session_open acct=""root"" exe=""/usr/sbin/cron"" hos...",1716,0,0,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,internal_share,audit.log,5,type=CRED_DISP msg=audit(1642724221.491:153): pid=1716 u...,CRED_DISP,1.642724e+09,153,2022-01-21 00:17:01.491000+00:00,"op=PAM:setcred acct=""root"" exe=""/usr/sbin/cron"" hostname...",1716,0,0,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Last 5 rows:


,source_host,source_log,line_number,raw_line,type,epoch,serial,timestamp,msg,pid,uid,auid,ses,old_auid,tty,old_ses,res,apparmor,operation,profile,name,comm,arch,syscall,success,exit,a0,a1,a2,a3,items,ppid,gid,euid,suid,fsuid,egid,sgid,fsgid,exe,key,proctitle
727,internal_share,audit.log,728,type=CRED_ACQ msg=audit(1643066221.955:868): pid=15520 u...,CRED_ACQ,1.643066e+09,868,2022-01-24 23:17:01.955000+00:00,"op=PAM:setcred acct=""root"" exe=""/usr/sbin/cron"" hostname...",15520,0,4294967295,4294967295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
728,internal_share,audit.log,729,type=LOGIN msg=audit(1643066221.959:869): pid=15520 uid=...,LOGIN,1.643066e+09,869,2022-01-24 23:17:01.959000+00:00,NaN,15520,0,0,141,4294967295,(none),4294967295,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
729,internal_share,audit.log,730,type=USER_START msg=audit(1643066221.963:870): pid=15520...,USER_START,1.643066e+09,870,2022-01-24 23:17:01.963000+00:00,"op=PAM:session_open acct=""root"" exe=""/usr/sbin/cron"" hos...",15520,0,0,141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
730,internal_share,audit.log,731,type=CRED_DISP msg=audit(1643066221.967:871): pid=15520 ...,CRED_DISP,1.643066e+09,871,2022-01-24 23:17:01.967000+00:00,"op=PAM:setcred acct=""root"" exe=""/usr/sbin/cron"" hostname...",15520,0,0,141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
731,internal_share,audit.log,732,type=USER_END msg=audit(1643066221.967:872): pid=15520 u...,USER_END,1.643066e+09,872,2022-01-24 23:17:01.967000+00:00,"op=PAM:session_close acct=""root"" exe=""/usr/sbin/cron"" ho...",15520,0,0,141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Ground Truth Labels

The label file contains 2 labeled lines (667-668) identifying exfiltration service activity.

In [24]:
import json

# Load labels
labels = []
with open(LABEL_FILE) as f:
    for entry in f:
        entry = entry.strip()
        if entry:
            labels.append(json.loads(entry))

print(f"Labeled lines: {len(labels)}")
print()
for lbl in labels:
    print(json.dumps(lbl, indent=2))
    print()

Labeled lines: 2

{
  "line": 667,
  "labels": [
    "dnsteal",
    "exfiltration-service",
    "attacker"
  ],
  "rules": {
    "dnsteal": [
      "exfil.service"
    ],
    "exfiltration-service": [
      "exfil.service"
    ],
    "attacker": [
      "exfil.service"
    ]
  }
}

{
  "line": 668,
  "labels": [
    "dnsteal",
    "exfiltration-service",
    "attacker"
  ],
  "rules": {
    "dnsteal": [
      "exfil.service"
    ],
    "exfiltration-service": [
      "exfil.service"
    ],
    "attacker": [
      "exfil.service"
    ]
  }
}



In [25]:
# Match labeled lines to parsed records
labeled_lines = {lbl["line"] for lbl in labels}
df_labeled = df[df["line_number"].isin(labeled_lines)].copy()

print(f"Matched {len(df_labeled)}/{len(labels)} labeled lines to parsed data\n")

# Show the labeled records with key fields
display_cols = ["line_number", "type", "timestamp", "pid", "uid", "auid", "ses", "msg"]
display_cols_present = [c for c in display_cols if c in df_labeled.columns]
for _, row in df_labeled.iterrows():
    print(f"Line {int(row['line_number'])}:")
    for col in display_cols_present:
        print(f"  {col}: {row[col]}")
    # Find corresponding label
    for lbl in labels:
        if lbl["line"] == int(row["line_number"]):
            print(f"  LABELS: {lbl['labels']}")
            print(f"  RULES: {lbl['rules']}")
    print()

Matched 2/2 labeled lines to parsed data

Line 667:
  line_number: 667
  type: SERVICE_START
  timestamp: 2022-01-24 13:50:39.298000+00:00
  pid: 1
  uid: 0
  auid: 4294967295
  ses: 4294967295
  msg: unit=put comm="systemd" exe="/lib/systemd/systemd" hostname=? addr=? terminal=? res=success
  LABELS: ['dnsteal', 'exfiltration-service', 'attacker']
  RULES: {'dnsteal': ['exfil.service'], 'exfiltration-service': ['exfil.service'], 'attacker': ['exfil.service']}

Line 668:
  line_number: 668
  type: SERVICE_STOP
  timestamp: 2022-01-24 13:50:39.298000+00:00
  pid: 1
  uid: 0
  auid: 4294967295
  ses: 4294967295
  msg: unit=put comm="systemd" exe="/lib/systemd/systemd" hostname=? addr=? terminal=? res=success
  LABELS: ['dnsteal', 'exfiltration-service', 'attacker']
  RULES: {'dnsteal': ['exfil.service'], 'exfiltration-service': ['exfil.service'], 'attacker': ['exfil.service']}



In [26]:
# Label distribution
all_labels = []
all_rules = []
for lbl in labels:
    all_labels.extend(lbl["labels"])
    for _rule_label, rule_names in lbl["rules"].items():
        for rule_name in rule_names:
            all_rules.append(rule_name)

print("=== Label Distribution ===")
for label_name, count in Counter(all_labels).most_common():
    print(f"  {label_name}: {count}")

print("\n=== Labels per line ===")
for lbl in labels:
    print(f"  Line {lbl['line']}: {len(lbl['labels'])} labels -> {lbl['labels']}")

print("\n=== Rules triggered ===")
for rule, count in Counter(all_rules).most_common():
    print(f"  {rule}: {count}")

=== Label Distribution ===
  dnsteal: 2
  exfiltration-service: 2
  attacker: 2

=== Labels per line ===
  Line 667: 3 labels -> ['dnsteal', 'exfiltration-service', 'attacker']
  Line 668: 3 labels -> ['dnsteal', 'exfiltration-service', 'attacker']

=== Rules triggered ===
  exfil.service: 6


In [27]:
# Show raw log lines for labeled events
print("=== Raw log lines for labeled events ===\n")
for lbl in labels:
    line_idx = lbl["line"] - 1  # 0-indexed
    print(f"[line {lbl['line']}] labels={lbl['labels']}")
    print(f"  {raw_lines[line_idx].rstrip()}")
    print()

=== Raw log lines for labeled events ===

[line 667] labels=['dnsteal', 'exfiltration-service', 'attacker']
  type=SERVICE_START msg=audit(1643032239.298:807): pid=1 uid=0 auid=4294967295 ses=4294967295 msg='unit=put comm="systemd" exe="/lib/systemd/systemd" hostname=? addr=? terminal=? res=success'

[line 668] labels=['dnsteal', 'exfiltration-service', 'attacker']
  type=SERVICE_STOP msg=audit(1643032239.298:808): pid=1 uid=0 auid=4294967295 ses=4294967295 msg='unit=put comm="systemd" exe="/lib/systemd/systemd" hostname=? addr=? terminal=? res=success'



## 6. Summary Statistics

In [28]:
print("=== Summary ===")
print(f"  Total lines: {len(df)}")
print(f"  Event types: {df['type'].nunique()}")
print(f"  Serials: {df['serial'].nunique()} unique")
print(f"  Time span: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
print(f"  Labeled lines: {len(labels)} ({len(labels) / len(df) * 100:.1f}%)")
print(f"  Raw table columns: {len(df_raw.columns)}")
print(f"  Analysis columns (with msg_*): {len(df.columns)}")
print()
print("Event breakdown:")
print(
    f"  Cron PAM events (root): {sum(1 for _, r in df.iterrows() if r.get('msg_acct') == 'root' and r.get('msg_exe') == '/usr/sbin/cron')}"
)
print(f"  Service events: {len(df[df['type'].isin({'SERVICE_START', 'SERVICE_STOP'})])}")
print(
    f"  Kernel events (AVC+SYSCALL+PROCTITLE): {len(df[df['type'].isin({'AVC', 'SYSCALL', 'PROCTITLE'})])}"
)
print(f"  Labeled lines: {len(labels)}")
print()
print("Key differences from intranet_server audit log (DAT-42):")
print("  - Fewer event types: 11 vs 15 (no USER_LOGIN, USER_AUTH, USER_CMD, CRED_REFR)")
print("  - Fewer lines: 732 vs 2,316")
print("  - Only root account (no jhall or www-data users)")
print("  - No SSH/network login events (all local cron/systemd)")
print("  - Attack type: exfiltration service (not privilege escalation)")
print("  - Only 2 labeled lines (vs 9 in intranet_server)")

=== Summary ===
  Total lines: 732
  Event types: 11
  Serials: 724 unique
  Time span: 2022-01-21 to 2022-01-24
  Labeled lines: 2 (0.3%)
  Raw table columns: 42
  Analysis columns (with msg_*): 50

Event breakdown:
  Cron PAM events (root): 530
  Service events: 84
  Kernel events (AVC+SYSCALL+PROCTITLE): 12
  Labeled lines: 2

Key differences from intranet_server audit log (DAT-42):
  - Fewer event types: 11 vs 15 (no USER_LOGIN, USER_AUTH, USER_CMD, CRED_REFR)
  - Fewer lines: 732 vs 2,316
  - Only root account (no jhall or www-data users)
  - No SSH/network login events (all local cron/systemd)
  - Attack type: exfiltration service (not privilege escalation)
  - Only 2 labeled lines (vs 9 in intranet_server)


## 7. Schema Mapping

### 7.0 Type Inference Assumptions

Same assumptions as DAT-42 (intranet_server audit log):

| Field | Assumed Type | Reasoning |
|-------|-------------|----------|
| `auid`, `ses`, `old_auid`, `old_ses` | BIGINT | Sentinel value 4294967295 exceeds INTEGER max (2,147,483,647) |
| `exit` | BIGINT | SYSCALL exit codes can be large/negative |
| `epoch` | DOUBLE PRECISION | Millisecond-precision float |
| `pid`, `uid`, `gid`, `euid`, `suid`, `fsuid`, `egid`, `sgid`, `fsgid`, `ppid`, `items`, `syscall` | INTEGER | Observed max values fit comfortably |
| `msg` | TEXT | Raw nested blob (1NF violation) |
| `proctitle` | TEXT | Hex-encoded command line, variable length |

### 7.1 Column Mapping Table

This file maps to the shared parsed staging table `stg_audit_line_raw`, the same shape as the intranet_server audit log (DAT-42). The `source_host` and `source_log` columns distinguish records from different audit sources. This file has fewer event types (11 vs 15), so some columns populated in the intranet_server log will be entirely NULL here. We keep the full schema for compatibility.

In [29]:
# Show column inventory for the raw table
print(f"Raw table columns ({len(df_raw.columns)}):\n")
print(f"{'#':<4} {'Column':<20} {'Non-null':>10} {'Null%':>8} {'Sample value':>40}")
print("-" * 85)
for i, col in enumerate(df_raw.columns, 1):
    non_null = df_raw[col].notna().sum()
    null_pct = df_raw[col].isna().sum() / len(df_raw) * 100
    sample = str(df_raw[col].dropna().iloc[0]) if non_null > 0 else "(all null)"
    if len(sample) > 38:
        sample = sample[:35] + "..."
    print(f"{i:<4} {col:<20} {non_null:>10} {null_pct:>7.1f}% {sample:>40}")

Raw table columns (42):

#    Column                 Non-null    Null%                             Sample value
-------------------------------------------------------------------------------------
1    source_host                 732     0.0%                           internal_share
2    source_log                  732     0.0%                                audit.log
3    line_number                 732     0.0%                                        1
4    raw_line                    732     0.0%   type=USER_ACCT msg=audit(1642724221...
5    type                        732     0.0%                                USER_ACCT
6    epoch                       732     0.0%                           1642724221.475
7    serial                      732     0.0%                                      149
8    timestamp                   732     0.0%         2022-01-21 00:17:01.475000+00:00
9    msg                         614    16.1%   op=PAM:accounting acct="root" exe="...
10   pid           

### 7.2 Parsed Staging DDL

This file maps to the shared staging table shape `stg_audit_line_raw`. The `source_host` and `source_log` columns distinguish records from different audit sources.

In [30]:
pg_ddl = """
-- PostgreSQL
CREATE TABLE stg_audit_line_raw (
    row_id          SERIAL PRIMARY KEY,
    source_host     VARCHAR(30) NOT NULL,
    source_log      VARCHAR(50) NOT NULL,
    line_number     INTEGER NOT NULL,
    raw_line        TEXT NOT NULL,
    type            VARCHAR(20) NOT NULL,
    epoch           DOUBLE PRECISION NOT NULL,
    serial          INTEGER NOT NULL,
    timestamp       TIMESTAMP WITH TIME ZONE NOT NULL,
    pid             INTEGER,
    uid             INTEGER,
    auid            BIGINT,
    ses             BIGINT,
    msg             TEXT,
    old_auid        BIGINT,
    old_ses         BIGINT,
    tty             VARCHAR(30),
    res             VARCHAR(10),
    apparmor        VARCHAR(20),
    operation       VARCHAR(30),
    info            TEXT,
    profile         VARCHAR(50),
    name            TEXT,
    comm            VARCHAR(50),
    exe             TEXT,
    arch            VARCHAR(20),
    syscall         INTEGER,
    success         VARCHAR(5),
    exit            BIGINT,
    a0              VARCHAR(20),
    a1              VARCHAR(20),
    a2              VARCHAR(20),
    a3              VARCHAR(20),
    items           INTEGER,
    ppid            INTEGER,
    gid             INTEGER,
    euid            INTEGER,
    suid            INTEGER,
    fsuid           INTEGER,
    egid            INTEGER,
    sgid            INTEGER,
    fsgid           INTEGER,
    key             VARCHAR(20),
    proctitle       TEXT
);
""".strip()

mysql_ddl = """
-- MySQL
CREATE TABLE stg_audit_line_raw (
    row_id          INT AUTO_INCREMENT PRIMARY KEY,
    source_host     VARCHAR(30) NOT NULL,
    source_log      VARCHAR(50) NOT NULL,
    line_number     INT NOT NULL,
    raw_line        TEXT NOT NULL,
    type            VARCHAR(20) NOT NULL,
    epoch           DOUBLE NOT NULL,
    serial          INT NOT NULL,
    timestamp       DATETIME NOT NULL,
    pid             INT,
    uid             INT,
    auid            BIGINT,
    ses             BIGINT,
    msg             TEXT,
    old_auid        BIGINT,
    old_ses         BIGINT,
    tty             VARCHAR(30),
    res             VARCHAR(10),
    apparmor        VARCHAR(20),
    operation       VARCHAR(30),
    info            TEXT,
    profile         VARCHAR(50),
    name            TEXT,
    comm            VARCHAR(50),
    exe             TEXT,
    arch            VARCHAR(20),
    syscall         INT,
    success         VARCHAR(5),
    exit            BIGINT,
    a0              VARCHAR(20),
    a1              VARCHAR(20),
    a2              VARCHAR(20),
    a3              VARCHAR(20),
    items           INT,
    ppid            INT,
    gid             INT,
    euid            INT,
    suid            INT,
    fsuid           INT,
    egid            INT,
    sgid            INT,
    fsgid           INT,
    `key`           VARCHAR(20),
    proctitle       TEXT
);
""".strip()

print(pg_ddl)
print()
print(mysql_ddl)

-- PostgreSQL
CREATE TABLE stg_audit_line_raw (
    row_id          SERIAL PRIMARY KEY,
    source_host     VARCHAR(30) NOT NULL,
    source_log      VARCHAR(50) NOT NULL,
    line_number     INTEGER NOT NULL,
    raw_line        TEXT NOT NULL,
    type            VARCHAR(20) NOT NULL,
    epoch           DOUBLE PRECISION NOT NULL,
    serial          INTEGER NOT NULL,
    timestamp       TIMESTAMP WITH TIME ZONE NOT NULL,
    pid             INTEGER,
    uid             INTEGER,
    auid            BIGINT,
    ses             BIGINT,
    msg             TEXT,
    old_auid        BIGINT,
    old_ses         BIGINT,
    tty             VARCHAR(30),
    res             VARCHAR(10),
    apparmor        VARCHAR(20),
    operation       VARCHAR(30),
    info            TEXT,
    profile         VARCHAR(50),
    name            TEXT,
    comm            VARCHAR(50),
    exe             TEXT,
    arch            VARCHAR(20),
    syscall         INTEGER,
    success         VARCHAR(5),
    exi

## 8. Normalization Observations

Applying the normalization rules checklist (Lecture 4 and 5).

### 8.1 1NF Check

**Multi-valued field identified:** The `msg` column packs multiple key-value pairs into a single TEXT blob (e.g., `op=PAM:accounting acct="root" exe="/usr/sbin/cron" hostname=? addr=? terminal=cron res=success`). This contains 7 distinct attributes (op, acct, exe, hostname, addr, terminal, res) for PAM events, and different attributes (unit, comm, exe, hostname, addr, terminal, res) for SERVICE events.

| Field | Violation | Values per row | Distinct sub-fields | Resolution direction |
|-------|-----------|---------------|--------------------|-----------------------|
| `msg` | Multi-valued (key-value pairs packed in TEXT blob) | 5-7 sub-fields per row | op, acct, exe, hostname, addr, terminal, res, unit, comm | Unpack into separate columns during normalization |

**1NF status: VIOLATED.**

### 8.2 2NF Check

The raw table uses a single-column surrogate primary key (`row_id`). Partial dependencies only arise with composite keys. No composite key exists in this raw table.

**2NF status: SATISFIED** (single-column PK; no composite key means no partial dependencies possible).

### 8.3 3NF Check

**Transitive dependency identified:** The `type` column determines which subset of the table's columns are populated.

- PAM events (USER_ACCT, CRED_ACQ, etc.) populate: pid, uid, auid, ses, msg (with op, acct, exe, hostname, addr, terminal, res)
- LOGIN events populate: pid, uid, old_auid, auid, tty, old_ses, ses, res
- SERVICE events populate: pid, uid, auid, ses, msg (with unit, comm, exe, hostname, addr, terminal, res)
- AVC events populate: pid, apparmor, operation, info, profile, name, comm
- SYSCALL events populate: arch, syscall, success, exit, a0-a3, items, ppid, pid, auid, uid, gid, euid, suid, fsuid, egid, sgid, fsgid, tty, ses, comm, exe, key
- PROCTITLE events populate: only proctitle

This is the pattern `row_id -> type -> {populated field set}`, a transitive dependency where a non-key attribute (`type`) determines other non-key attributes.

**3NF status: structural heterogeneity noted.** The type -> field_set correlation may motivate subtype modeling in the final design (e.g., audit_pam_events, audit_login_events, audit_service_events, audit_kernel_events).

### 8.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | `row_id` | all other attributes | Surrogate PK. |
| FD2 | (source_host, source_log, line_number) | all other attributes | Each line in a given source file is unique. Staging candidate key across files. |
| FD3 | `type` | populated field set | Each event type uses a specific subset of columns (structural heterogeneity, normalization input). |
| FD4 | `serial` | `epoch`, `timestamp` | All rows with the same serial share the same timestamp (multi-line events). |
| FD5 | `old_auid` | constant `4294967295` | All 106 LOGIN events have old_auid = 4294967295. Trivial (only one value). |

## 9. Key Findings for Schema Design

1. **Simpler than intranet_server:** 11 event types (vs 15), 732 lines (vs 2,316). No SSH login, no privilege escalation, no user accounts besides root. The internal_share is a quiet file server.

2. **Same auditd format:** Identical two-level structure (outer fields + nested `msg='...'` blob). The same parser handles both files. The staging table schema shares the same column set.

3. **1NF violation (msg column):** Same as intranet_server. The `msg` TEXT column packs multiple key-value pairs. Resolution: unpack into separate columns during normalization.

4. **Structural heterogeneity (type -> field_set):** Same pattern as intranet_server. The 11 event types each use a different subset of columns, which may motivate subtype modeling in the final design.

5. **Exfiltration service (labeled lines):** Only 2 labeled lines (667-668). A SERVICE_START and SERVICE_STOP for `unit=put`, labeled as `dnsteal`, `exfiltration-service`, `attacker`. This is the attacker starting/stopping a data exfiltration service on the file server.

6. **Dominant pattern is cron:** ~86.9% of events are root cron PAM cycles (6-event sequences: USER_ACCT, CRED_ACQ, LOGIN, USER_START, CRED_DISP, USER_END). Highly predictable baseline.

7. **Multi-line events:** 4 serials (193-196) span 3 lines each (AVC + SYSCALL + PROCTITLE). All are AppArmor profile_replace operations for dhclient, not attack-related.

8. **No network addresses:** All hostname=?, addr=?. No SSH logins, no remote access events. The attacker's IP (172.19.131.174) does NOT appear in this log - the exfiltration service was started via a mechanism not captured by auditd on this host.

9. **Cross-file synthesis with intranet_server (DAT-42):** This file shares the auditd source format with the intranet audit log and will be reconciled in the final audit schema design. The intranet_server has the privilege escalation events; the internal_share has the exfiltration service events.